In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Labs"
)

xpt_file = DATA_DIR / "GLU_H.xpt"
csv_file = DATA_DIR / "GLU_H_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2013-2014 GLU_H XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WTSAF2YR",
    "LBXGLU",
    "LBDGLUSI",
    "PHAFSTHR",
    "PHAFSTMN"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "GLU_H column names/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 3329:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 6:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 3,329 rows x 6 columns."
)


# ============================================================
# 4. DEFINE FIELD TYPES
# ============================================================

#
# GLU_H is a MIXED integer / decimal Laboratory file.
#
#
# SEQN
#     Respondent sequence number
#     -> whole-number identifier
#
#
# WTSAF2YR
#     Fasting subsample 2-year MEC weight
#     -> genuine continuous decimal
#
#
# LBXGLU
#     Fasting glucose in mg/dL
#     -> released as whole-number values
#
#
# LBDGLUSI
#     Fasting glucose in mmol/L
#     -> genuine continuous decimal
#
#
# PHAFSTHR
#     Total fasting hours
#     -> whole-number
#
#
# PHAFSTMN
#     Total fasting minutes
#     -> whole-number
#
#
# Therefore:
#
#     Int64:
#         SEQN
#         LBXGLU
#         PHAFSTHR
#         PHAFSTMN
#
#     KEEP FLOAT:
#         WTSAF2YR
#         LBDGLUSI
#
#
# IMPORTANT:
#
# Genuine continuous decimal values remain EXACTLY AS IS.
#
# DO NOT:
#
#     round()
#     globally remove ".0"
#     use float_format=
#     convert genuine decimals to Int64
#

DECIMAL_COLUMNS = [
    "WTSAF2YR",
    "LBDGLUSI"
]


INTEGER_COLUMNS = [
    "SEQN",
    "LBXGLU",
    "PHAFSTHR",
    "PHAFSTMN"
]


if len(DECIMAL_COLUMNS) != 2:

    raise ValueError(
        "Expected exactly 2 genuine decimal fields."
    )


if len(INTEGER_COLUMNS) != 4:

    raise ValueError(
        "Expected exactly 4 whole-number fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN SAS XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT representation of numeric zero:
#
#     5.397605346934028e-79
#
#
# We NEVER globally replace it before confirming the
# complete variable/count pattern.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 6 GLU_H fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


tiny_variable_count = int(
    tiny_counts.gt(0)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    tiny_variable_count
)


print(
    "\nTiny values by variable:"
)

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Direct inspection of attached GLU_H.xpt:
#
#     WTSAF2YR     402
#     PHAFSTHR      64
#     PHAFSTMN      55
#
#     TOTAL        521
#
#
# CDC confirms:
#
# WTSAF2YR:
#     positive = 2,927
#     zero     =   402
#
# PHAFSTHR:
#     allowed range includes 0
#
# PHAFSTMN:
#     allowed range includes 0
#

EXPECTED_TINY_COUNTS = {
    "WTSAF2YR": 402,
    "PHAFSTHR": 64,
    "PHAFSTMN": 55
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated GLU_H attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 521:

    raise ValueError(
        f"Unexpected tiny-value total: "
        f"{tiny_total:,}"
    )


if tiny_variable_count != 3:

    raise ValueError(
        "Expected tiny values in exactly 3 variables."
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary zeros existed "
        "before tiny-value restoration."
    )


print(
    "\nPASS: Exact 521-value tiny-zero pattern "
    "across 3 variables validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#

print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 521:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# This becomes the exact fidelity reference.
#
# The ONLY permitted numerical modifications are:
#
#     validated tiny -> 0
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# Run BEFORE Int64 conversion.
#
# np.round() is used ONLY to test whether a number
# contains a fractional component.
#
# No dataframe value is rounded.
#

print(
    "\n--- GENUINE DECIMAL DETECTION ---"
)


detected_decimal_columns = []

fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = fractional_count


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        detected_decimal_columns.append(
            col
        )


if detected_decimal_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Genuine decimal-variable pattern differs.\n\n"
        f"Expected:\n{DECIMAL_COLUMNS}\n\n"
        f"Actual:\n{detected_decimal_columns}"
    )


print(
    "\nPASS: Exactly 2 genuine decimal fields detected."
)


# ============================================================
# 11. DECIMAL FIELD DETAILS
# ============================================================

print(
    "\n--- DECIMAL FIELD DETAILS ---"
)


for col in DECIMAL_COLUMNS:

    s = df[col]


    nonmissing = int(
        s.notna()
        .sum()
    )


    missing = int(
        s.isna()
        .sum()
    )


    fractional = fractional_counts[col]


    whole_looking = (
        nonmissing
        -
        fractional
    )


    print(
        f"\n{col}"
    )

    print(
        "  Nonmissing:",
        f"{nonmissing:,}"
    )

    print(
        "  Missing:",
        f"{missing:,}"
    )

    print(
        "  Minimum:",
        repr(float(s.min()))
    )

    print(
        "  Maximum:",
        repr(float(s.max()))
    )

    print(
        "  Fractional observations:",
        f"{fractional:,}"
    )

    print(
        "  Whole-looking observations:",
        f"{whole_looking:,}"
    )


# ============================================================
# 12. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 73559:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 83727:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 73559-83727 validated."
)


# ============================================================
# 13. VALIDATE LBXGLU IS WHOLE-NUMBER
# ============================================================

glu_values = (
    df["LBXGLU"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    glu_values,
    np.round(glu_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "LBXGLU unexpectedly contains "
        "fractional values."
    )


print(
    "PASS: Every populated LBXGLU value "
    "is whole-number."
)


# ============================================================
# 14. VALIDATE FASTING TIME FIELDS ARE WHOLE-NUMBER
# ============================================================

for col in [
    "PHAFSTHR",
    "PHAFSTMN"
]:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if not np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    ).all():

        raise ValueError(
            f"{col} unexpectedly contains "
            "fractional values."
        )


print(
    "PASS: Fasting hour/minute fields "
    "contain only whole-number values."
)


# ============================================================
# 15. CONVERT PROVEN WHOLE-NUMBER FIELDS TO Int64
# ============================================================

#
# NO .round() during actual conversion.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: SEQN, LBXGLU, PHAFSTHR and PHAFSTMN "
    "converted to nullable Int64 without rounding."
)


# ============================================================
# 16. CONFIRM DECIMAL FIELDS REMAIN FLOAT
# ============================================================

decimal_dtype_failures = []


for col in DECIMAL_COLUMNS:

    if not pd.api.types.is_float_dtype(
        df[col].dtype
    ):

        decimal_dtype_failures.append(
            col
        )


if decimal_dtype_failures:

    raise ValueError(
        "Continuous variables are no longer float:\n"
        + str(
            decimal_dtype_failures
        )
    )


print(
    "PASS: WTSAF2YR and LBDGLUSI "
    "remain floating-point."
)


# ============================================================
# 17. CDC / XPT CHECK - WTSAF2YR
# ============================================================

#
# CDC HTML displays:
#
#     12937.987305 to 395978.46579
#
#
# Actual attached XPT stores maximum:
#
#     395978.465792
#
#
# The XPT value has slightly greater precision.
#
# Preserve the XPT exactly.
#

weight_positive = int(
    df["WTSAF2YR"]
    .gt(0)
    .sum()
)


weight_zero = int(
    df["WTSAF2YR"]
    .eq(0)
    .sum()
)


weight_missing = int(
    df["WTSAF2YR"]
    .isna()
    .sum()
)


positive_weights = (
    df.loc[
        df["WTSAF2YR"] > 0,
        "WTSAF2YR"
    ]
)


weight_min = float(
    positive_weights.min()
)


weight_max = float(
    positive_weights.max()
)


print(
    "\n--- WTSAF2YR ---"
)

print(
    "Positive:",
    f"{weight_positive:,}"
)

print(
    "Zero:",
    f"{weight_zero:,}"
)

print(
    "Missing:",
    f"{weight_missing:,}"
)

print(
    "Exact XPT minimum:",
    repr(weight_min)
)

print(
    "Exact XPT maximum:",
    repr(weight_max)
)


if weight_positive != 2927:

    raise ValueError(
        "WTSAF2YR positive count differs."
    )


if weight_zero != 402:

    raise ValueError(
        "WTSAF2YR zero count differs."
    )


if weight_missing != 0:

    raise ValueError(
        "Unexpected missing WTSAF2YR."
    )


if weight_min != 12937.987305:

    raise ValueError(
        "WTSAF2YR exact XPT minimum differs."
    )


if weight_max != 395978.465792:

    raise ValueError(
        "WTSAF2YR exact XPT maximum differs."
    )


#
# Secondary CDC DISPLAY checks only.
#
# These round temporary scalars.
#
# They DO NOT modify the dataframe.
#

if round(
    weight_min,
    6
) != 12937.987305:

    raise ValueError(
        "WTSAF2YR minimum does not match "
        "CDC displayed precision."
    )


if round(
    weight_max,
    5
) != 395978.46579:

    raise ValueError(
        "WTSAF2YR maximum does not match "
        "CDC displayed precision."
    )


print(
    "PASS: WTSAF2YR exact XPT precision preserved."
)

print(
    "PASS: WTSAF2YR also matches "
    "CDC displayed range."
)


# ============================================================
# 18. CDC CHECK - LBXGLU
# ============================================================

#
# CDC:
#
#     Range:
#         51-421 mg/dL
#
#     Nonmissing:
#         3,172
#
#     Missing:
#         157
#

glu_nonmissing = int(
    df["LBXGLU"]
    .notna()
    .sum()
)


glu_missing = int(
    df["LBXGLU"]
    .isna()
    .sum()
)


glu_range = int(
    df["LBXGLU"]
    .between(
        51,
        421
    )
    .sum()
)


glu_min = int(
    df["LBXGLU"]
    .min()
)


glu_max = int(
    df["LBXGLU"]
    .max()
)


print(
    "\n--- LBXGLU ---"
)

print(
    "Nonmissing:",
    f"{glu_nonmissing:,}"
)

print(
    "Missing:",
    f"{glu_missing:,}"
)

print(
    "Minimum:",
    glu_min
)

print(
    "Maximum:",
    glu_max
)


if glu_nonmissing != 3172:

    raise ValueError(
        "LBXGLU nonmissing count differs."
    )


if glu_missing != 157:

    raise ValueError(
        "LBXGLU missing count differs."
    )


if glu_range != 3172:

    raise ValueError(
        "LBXGLU contains values outside 51-421."
    )


if glu_min != 51:

    raise ValueError(
        "LBXGLU minimum differs."
    )


if glu_max != 421:

    raise ValueError(
        "LBXGLU maximum differs."
    )


print(
    "PASS: LBXGLU matches CDC range/count."
)


# ============================================================
# 19. CDC CHECK - LBDGLUSI
# ============================================================

#
# CDC:
#
#     Range:
#         2.831-23.37 mmol/L
#
#     Nonmissing:
#         3,172
#
#     Missing:
#         157
#

si_nonmissing = int(
    df["LBDGLUSI"]
    .notna()
    .sum()
)


si_missing = int(
    df["LBDGLUSI"]
    .isna()
    .sum()
)


si_range = int(
    df["LBDGLUSI"]
    .between(
        2.831,
        23.37
    )
    .sum()
)


si_min = float(
    df["LBDGLUSI"]
    .min()
)


si_max = float(
    df["LBDGLUSI"]
    .max()
)


print(
    "\n--- LBDGLUSI ---"
)

print(
    "Nonmissing:",
    f"{si_nonmissing:,}"
)

print(
    "Missing:",
    f"{si_missing:,}"
)

print(
    "Minimum:",
    repr(si_min)
)

print(
    "Maximum:",
    repr(si_max)
)


if si_nonmissing != 3172:

    raise ValueError(
        "LBDGLUSI nonmissing count differs."
    )


if si_missing != 157:

    raise ValueError(
        "LBDGLUSI missing count differs."
    )


if si_range != 3172:

    raise ValueError(
        "LBDGLUSI contains values outside "
        "CDC range."
    )


if si_min != 2.831:

    raise ValueError(
        "LBDGLUSI minimum differs."
    )


if si_max != 23.37:

    raise ValueError(
        "LBDGLUSI maximum differs."
    )


print(
    "PASS: LBDGLUSI matches CDC range/count."
)


# ============================================================
# 20. LBXGLU / LBDGLUSI MISSING MASK CHECK
# ============================================================

glu_missing_mask = (
    df["LBXGLU"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


si_missing_mask = (
    df["LBDGLUSI"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


glucose_mask_mismatch = int(
    np.sum(
        glu_missing_mask
        !=
        si_missing_mask
    )
)


print(
    "\n--- GLUCOSE MISSING MASK CHECK ---"
)

print(
    "Mismatches:",
    glucose_mask_mismatch
)


if glucose_mask_mismatch != 0:

    raise ValueError(
        "LBXGLU and LBDGLUSI missing "
        "positions differ."
    )


print(
    "PASS: LBXGLU and LBDGLUSI missing "
    "positions match exactly."
)


# ============================================================
# 21. CDC GLUCOSE SI CONVERSION CHECK
# ============================================================

#
# CDC documentation:
#
#     LBDGLUSI = LBXGLU * 0.05551
#
# rounded to 3 decimals.
#
#
# Decimal + ROUND_HALF_UP is used ONLY for validation.
#
# We NEVER regenerate or overwrite LBDGLUSI.
#
# The released XPT value remains the fidelity source.
#

valid_glucose_mask = (
    df["LBXGLU"]
    .notna()
    &
    df["LBDGLUSI"]
    .notna()
)


conversion_mismatches = 0


for mg_dl, released_si in zip(
    df.loc[
        valid_glucose_mask,
        "LBXGLU"
    ],
    df.loc[
        valid_glucose_mask,
        "LBDGLUSI"
    ]
):

    calculated = (
        Decimal(
            str(int(mg_dl))
        )
        *
        Decimal("0.05551")
    )


    calculated_3dp = calculated.quantize(
        Decimal("0.001"),
        rounding=ROUND_HALF_UP
    )


    released_3dp = Decimal(
        str(float(released_si))
    ).quantize(
        Decimal("0.001")
    )


    if calculated_3dp != released_3dp:

        conversion_mismatches += 1


print(
    "\n--- GLUCOSE SI CONVERSION CHECK ---"
)

print(
    "Records checked:",
    f"{int(valid_glucose_mask.sum()):,}"
)

print(
    "Conversion mismatches:",
    conversion_mismatches
)


if conversion_mismatches != 0:

    raise ValueError(
        "LBDGLUSI does not match documented "
        "LBXGLU x 0.05551 conversion."
    )


print(
    "PASS: All 3,172 LBDGLUSI values match "
    "the documented CDC conversion."
)


# ============================================================
# 22. CDC CHECK - PHAFSTHR
# ============================================================

#
# CDC:
#
#     Range:
#         0-34 hours
#
#     Nonmissing:
#         3,291
#
#     Missing:
#         38
#
#
# Direct XPT:
#
#     legitimate zeros = 64
#

fast_hr_nonmissing = int(
    df["PHAFSTHR"]
    .notna()
    .sum()
)


fast_hr_missing = int(
    df["PHAFSTHR"]
    .isna()
    .sum()
)


fast_hr_range = int(
    df["PHAFSTHR"]
    .between(
        0,
        34
    )
    .sum()
)


fast_hr_zero = int(
    df["PHAFSTHR"]
    .eq(0)
    .sum()
)


print(
    "\n--- PHAFSTHR ---"
)

print(
    "Nonmissing:",
    f"{fast_hr_nonmissing:,}"
)

print(
    "Missing:",
    f"{fast_hr_missing:,}"
)

print(
    "Zero:",
    f"{fast_hr_zero:,}"
)


if fast_hr_nonmissing != 3291:

    raise ValueError(
        "PHAFSTHR nonmissing count differs."
    )


if fast_hr_missing != 38:

    raise ValueError(
        "PHAFSTHR missing count differs."
    )


if fast_hr_range != 3291:

    raise ValueError(
        "PHAFSTHR values outside 0-34."
    )


if fast_hr_zero != 64:

    raise ValueError(
        "PHAFSTHR zero count differs."
    )


print(
    "PASS: PHAFSTHR range/count validated."
)


# ============================================================
# 23. CDC CHECK - PHAFSTMN
# ============================================================

#
# CDC:
#
#     Range:
#         0-59 minutes
#
#     Nonmissing:
#         3,291
#
#     Missing:
#         38
#
#
# Direct XPT:
#
#     legitimate zeros = 55
#

fast_mn_nonmissing = int(
    df["PHAFSTMN"]
    .notna()
    .sum()
)


fast_mn_missing = int(
    df["PHAFSTMN"]
    .isna()
    .sum()
)


fast_mn_range = int(
    df["PHAFSTMN"]
    .between(
        0,
        59
    )
    .sum()
)


fast_mn_zero = int(
    df["PHAFSTMN"]
    .eq(0)
    .sum()
)


print(
    "\n--- PHAFSTMN ---"
)

print(
    "Nonmissing:",
    f"{fast_mn_nonmissing:,}"
)

print(
    "Missing:",
    f"{fast_mn_missing:,}"
)

print(
    "Zero:",
    f"{fast_mn_zero:,}"
)


if fast_mn_nonmissing != 3291:

    raise ValueError(
        "PHAFSTMN nonmissing count differs."
    )


if fast_mn_missing != 38:

    raise ValueError(
        "PHAFSTMN missing count differs."
    )


if fast_mn_range != 3291:

    raise ValueError(
        "PHAFSTMN values outside 0-59."
    )


if fast_mn_zero != 55:

    raise ValueError(
        "PHAFSTMN zero count differs."
    )


print(
    "PASS: PHAFSTMN range/count validated."
)


# ============================================================
# 24. FASTING HOUR/MINUTE MISSING-MASK CHECK
# ============================================================

fast_hr_missing_mask = (
    df["PHAFSTHR"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


fast_mn_missing_mask = (
    df["PHAFSTMN"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


fasting_mask_mismatch = int(
    np.sum(
        fast_hr_missing_mask
        !=
        fast_mn_missing_mask
    )
)


print(
    "\n--- FASTING TIME MISSING MASK ---"
)

print(
    "Mismatches:",
    fasting_mask_mismatch
)


if fasting_mask_mismatch != 0:

    raise ValueError(
        "PHAFSTHR and PHAFSTMN missing "
        "positions differ."
    )


print(
    "PASS: Fasting hour/minute missing "
    "positions match exactly."
)


# ============================================================
# 25. WTSAF2YR / GLUCOSE AVAILABILITY PATTERN
# ============================================================

#
# IMPORTANT:
#
# WTSAF2YR = 0 does NOT mean every laboratory field
# must necessarily be missing.
#
# Preserve the released attachment exactly.
#
#
# Direct attachment pattern:
#
# positive weight + glucose present = 2,927
#
# zero weight + glucose missing      =   157
#
# zero weight + glucose present      =   245
#

positive_weight_glucose_present = int(
    (
        df["WTSAF2YR"].gt(0)
        &
        df["LBXGLU"].notna()
    )
    .sum()
)


zero_weight_glucose_missing = int(
    (
        df["WTSAF2YR"].eq(0)
        &
        df["LBXGLU"].isna()
    )
    .sum()
)


zero_weight_glucose_present = int(
    (
        df["WTSAF2YR"].eq(0)
        &
        df["LBXGLU"].notna()
    )
    .sum()
)


print(
    "\n--- WEIGHT / GLUCOSE AVAILABILITY ---"
)

print(
    "Positive weight + glucose present:",
    positive_weight_glucose_present
)

print(
    "Zero weight + glucose missing:",
    zero_weight_glucose_missing
)

print(
    "Zero weight + glucose present:",
    zero_weight_glucose_present
)


if positive_weight_glucose_present != 2927:

    raise ValueError(
        "Unexpected positive-weight/glucose pattern."
    )


if zero_weight_glucose_missing != 157:

    raise ValueError(
        "Unexpected zero-weight/glucose-missing pattern."
    )


if zero_weight_glucose_present != 245:

    raise ValueError(
        "Unexpected zero-weight/glucose-present pattern."
    )


print(
    "PASS: Released weight/result availability "
    "pattern preserved."
)


# ============================================================
# 26. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Original ordinary numeric zero count = 0.
#
# Therefore all zeros after repair should correspond
# exactly to the validated tiny positions.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO PATTERN ---"
)

print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


if sum(
    final_zero_counts.values()
) != 521:

    raise ValueError(
        "Unexpected final zero total."
    )


print(
    "PASS: Final zero pattern exactly matches "
    "all 521 validated restorations."
)


# ============================================================
# 27. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# ONLY the validated tiny -> zero repairs are allowed.
#
# Every other value must remain numerically identical.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: All 6 GLU_H variables match "
    "the exact corrected XPT values."
)


# ============================================================
# 28. EXPORT CSV
# ============================================================

#
# CRITICAL FIDELITY EXPORT:
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
#
# Whole-number fields:
#
#     SEQN
#     LBXGLU
#     PHAFSTHR
#     PHAFSTMN
#
# lose unnecessary ".0".
#
#
# Genuine decimal fields:
#
#     WTSAF2YR
#     LBDGLUSI
#
# remain exactly AS IS.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 29. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 30. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: 0 numeric differences across "
    "all 6 GLU_H variables."
)


# ============================================================
# 31. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 32. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 33. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 34. .0 CHECK - INTEGER FIELDS ONLY
# ============================================================

#
# VERY IMPORTANT:
#
# Do NOT globally check every column.
#
# WTSAF2YR and LBDGLUSI are genuine continuous fields.
#
# Only:
#
#     SEQN
#     LBXGLU
#     PHAFSTHR
#     PHAFSTMN
#
# should be checked for unnecessary ".0".
#

print(
    "\n--- INTEGER FIELD .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' in whole-number fields:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' in "
    "SEQN, LBXGLU, PHAFSTHR or PHAFSTMN."
)


# ============================================================
# 35. LEGITIMATE CONTINUOUS .0 VALUES
# ============================================================

#
# INFORMATIONAL / FIDELITY CHECK.
#
# Direct attachment:
#
# WTSAF2YR:
#
#     402 validated zero weights become:
#
#         0.0
#
#
# LBDGLUSI:
#
#     all 3,172 populated observations contain
#     a genuine fractional component, so this file
#     has no LBDGLUSI tokens ending in ".0".
#

print(
    "\n--- LEGITIMATE DECIMAL .0 TOKENS ---"
)


decimal_dot_zero_counts = {}


for col in DECIMAL_COLUMNS:

    count = int(
        (
            raw_csv_df[col].ne("")
            &
            raw_csv_df[col].str.endswith(".0")
        )
        .sum()
    )


    decimal_dot_zero_counts[col] = count


    print(
        f"{col}:",
        f"{count:,}"
    )


if decimal_dot_zero_counts["WTSAF2YR"] != 402:

    raise ValueError(
        "Unexpected WTSAF2YR '.0' token count."
    )


if decimal_dot_zero_counts["LBDGLUSI"] != 0:

    raise ValueError(
        "Unexpected LBDGLUSI '.0' token count."
    )


print(
    "PASS: Continuous decimal formatting preserved."
)


# ============================================================
# 36. EXACT DECIMAL FIDELITY
# ============================================================

#
# Most important continuous-value test:
#
#     rtol = 0
#     atol = 0
#
# No numerical difference is accepted.
#

print(
    "\n--- EXACT CONTINUOUS DECIMAL FIDELITY ---"
)


decimal_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        decimal_failures.append(
            col
        )


if decimal_failures:

    raise ValueError(
        "Continuous-value fidelity failed:\n"
        + str(
            decimal_failures
        )
    )


print(
    "PASS: WTSAF2YR and LBDGLUSI "
    "preserved exactly."
)


# ============================================================
# 37. FINAL WTSAF2YR PRECISION CHECK
# ============================================================

final_weight = pd.to_numeric(
    raw_csv_df["WTSAF2YR"]
    .where(
        raw_csv_df["WTSAF2YR"] != "",
        np.nan
    ),
    errors="coerce"
)


positive_final_weight = (
    final_weight[
        final_weight > 0
    ]
)


final_weight_min = float(
    positive_final_weight.min()
)


final_weight_max = float(
    positive_final_weight.max()
)


print(
    "\n--- FINAL WTSAF2YR PRECISION ---"
)

print(
    "Minimum:",
    repr(final_weight_min)
)

print(
    "Maximum:",
    repr(final_weight_max)
)


if final_weight_min != 12937.987305:

    raise ValueError(
        "WTSAF2YR minimum changed during export."
    )


if final_weight_max != 395978.465792:

    raise ValueError(
        "WTSAF2YR maximum changed during export."
    )


print(
    "PASS: WTSAF2YR full XPT precision preserved."
)


# ============================================================
# 38. FINAL GLUCOSE RANGE CHECK
# ============================================================

final_glu = pd.to_numeric(
    raw_csv_df["LBXGLU"]
    .where(
        raw_csv_df["LBXGLU"] != "",
        np.nan
    ),
    errors="coerce"
)


final_si = pd.to_numeric(
    raw_csv_df["LBDGLUSI"]
    .where(
        raw_csv_df["LBDGLUSI"] != "",
        np.nan
    ),
    errors="coerce"
)


if final_glu.min() != 51:

    raise ValueError(
        "Final LBXGLU minimum changed."
    )


if final_glu.max() != 421:

    raise ValueError(
        "Final LBXGLU maximum changed."
    )


if final_si.min() != 2.831:

    raise ValueError(
        "Final LBDGLUSI minimum changed."
    )


if final_si.max() != 23.37:

    raise ValueError(
        "Final LBDGLUSI maximum changed."
    )


print(
    "PASS: Final glucose ranges preserved."
)


# ============================================================
# 39. FINAL CSV ZERO PATTERN
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "the 521 validated zero values."
)


# ============================================================
# 40. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 41. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 73559:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 83727:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found in final CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 42. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "GLU_H CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2013-2014"
)


print(
    "Component: Laboratory - Plasma Fasting Glucose"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Tiny artifact variables:",
    list(
        EXPECTED_TINY_COUNTS.keys()
    )
)


print(
    "Validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Whole-number fields:",
    INTEGER_COLUMNS
)


print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


print(
    "WTSAF2YR positive:",
    f"{weight_positive:,}"
)


print(
    "WTSAF2YR zero:",
    f"{weight_zero:,}"
)


print(
    "WTSAF2YR exact minimum:",
    repr(weight_min)
)


print(
    "WTSAF2YR exact maximum:",
    repr(weight_max)
)


print(
    "LBXGLU nonmissing:",
    f"{glu_nonmissing:,}"
)


print(
    "LBXGLU missing:",
    f"{glu_missing:,}"
)


print(
    "LBXGLU range:",
    f"{glu_min}-{glu_max} mg/dL"
)


print(
    "LBDGLUSI nonmissing:",
    f"{si_nonmissing:,}"
)


print(
    "LBDGLUSI missing:",
    f"{si_missing:,}"
)


print(
    "LBDGLUSI range:",
    f"{si_min}-{si_max} mmol/L"
)


print(
    "PHAFSTHR zero:",
    fast_hr_zero
)


print(
    "PHAFSTMN zero:",
    fast_mn_zero
)


print(
    "Glucose SI conversion mismatches:",
    conversion_mismatches
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "Continuous-value differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Unnecessary '.0' in integer fields:",
    "0 expected"
)


print(
    "Continuous '.0' values:",
    "ALLOWED / PRESERVED"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "WTSAF2YR and LBDGLUSI remain "
    "genuine continuous decimal variables."
)


print(
    "No continuous Laboratory value was rounded "
    "or rewritten."
)

NHANES 2013-2014 GLU_H XPT -> CSV
Rows: 3,329
Columns: 6
PASS: Structure = 3,329 rows x 6 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: ['WTSAF2YR', 'LBDGLUSI']
Whole-number fields: ['SEQN', 'LBXGLU', 'PHAFSTHR', 'PHAFSTMN']

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 521
Variables containing tiny artifacts: 3

Tiny values by variable:
WTSAF2YR    402
PHAFSTHR     64
PHAFSTMN     55

PASS: Exact 521-value tiny-zero pattern across 3 variables validated.

--- VALIDATED ZERO RESTORATION ---
WTSAF2YR: 402 restored to 0
PHAFSTHR: 64 restored to 0
PHAFSTMN: 55 restored to 0

PASS: 521 validated zeros restored.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- GENUINE DECIMAL DETECTION ---
SEQN: 0 fractional observations
WTSAF2YR: 2,927 fractional observations
LBXGLU: 0 fractional observations
LBDGLUSI: 3,172 fractional observations
PHAFSTHR: 0 fractional observations
PHAFSTMN: 0 fractional observations

PAS